<a href="https://colab.research.google.com/github/Grenki-with-cheese/dissertation-notebook-2213935-cn6000/blob/main/notebook05_Recover_prompts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The notebook closely follows algorithms written by Hsu et al. (2024)

In [1]:
!pip install diffusers==0.30.0 transformers==4.44.0 accelerate==0.33.0 \ numpy==1.26.4 "scipy<1.14" -q
print("Install complete.")
print("If Cell 3 throws a numpy ABI error: Runtime > Restart session + run all")

Install complete.
If Cell 3 throws a numpy ABI error: Runtime > Restart session + run all


In [2]:
# === Cell 2: Boilerplate ===
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/MyDissertationCN6000')
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import torch
assert torch.cuda.is_available(), "Need a GPU runtime (A100 or L4)."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


Writing 30 paired prompts into a csv file, written by Claude using simple algorithm of generic prompts meant to reflect diversity of target prompts with added "with Vincent van Gogh style".

Following methodology by Hsu et al. (2024, Append H)

In [3]:
import pandas as pd
from pathlib import Path

pairs = [(f"{s}, with Vincent van Gogh style", s) for s in [
    "a self-portrait of a man with red hair and a beard",
    "a portrait of a young woman",
    "a portrait of an old peasant farmer",
    "a portrait of a postman in uniform",
    "a portrait of a young boy in a straw hat",
    "a portrait of a doctor with a melancholy expression",
    "a portrait of a woman in a green dress",
    "a self-portrait of a man with a felt hat",
    "a vase of yellow sunflowers",
    "a vase of irises in a garden",
    "a still life of pears and a wine bottle",
    "a bouquet of poppies and daisies",
    "a still life of shoes on a wooden floor",
    "an almond branch in blossom against a blue sky",
    "a chair in a small bedroom",
    "a still life of fruit on a table",
    "a wheat field under a stormy sky",
    "olive trees on a hillside",
    "a row of cypresses against a blue sky",
    "a country road lined with poplars",
    "rolling green hills under a yellow sky",
    "a flowering garden in spring",
    "a peasant sowing seeds in a field at sunset",
    "a meadow with bright red poppies",
    "a starry night over a small village with a tall cypress",
    "a café terrace at night",
    "a wooden drawbridge over a canal",
    "a river at night reflecting starlight",
    "a yellow house on a sunny street",
    "a small town under a blue evening sky",
]]
assert len(pairs) == 30

df = pd.DataFrame(pairs, columns=["van_gogh", "not_gogh"])
df.to_csv("prompts/vangogh_pairs.csv", index=False)
print(f"Wrote {len(df)} pairs to prompts/vangogh_pairs.csv")
print(df.head(3))

Wrote 30 pairs to prompts/vangogh_pairs.csv
                                            van_gogh  \
0  a self-portrait of a man with red hair and a b...   
1  a portrait of a young woman, with Vincent van ...   
2  a portrait of an old peasant farmer, with Vinc...   

                                            not_gogh  
0  a self-portrait of a man with red hair and a b...  
1                        a portrait of a young woman  
2                a portrait of an old peasant farmer  


Get concept vector

In [4]:
import numpy as np
import pandas as pd
from transformers import CLIPTextModel, CLIPTokenizer

# Same text encoder as the generation pipeline — ensures embedding-space alignment
tokenizer = CLIPTokenizer.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="text_encoder").to("cuda")

num_samples = 5
df = pd.read_csv("prompts/vangogh_pairs.csv")

def embed_prompts(prompt_list):
    """Embed each prompt num_samples times; return concatenated array."""
    out = []
    for prompt in prompt_list:
        prompts = [prompt] * num_samples
        text_input = tokenizer(prompts, padding="max_length", max_length=77,
                               truncation=True, return_tensors="pt")
        with torch.no_grad():
            embed = text_encoder(text_input.input_ids.to("cuda"), return_dict=True)[0]
        out.extend(embed.cpu().numpy())
    return np.array(out)

van_gogh_emb = embed_prompts(df["van_gogh"].tolist())
not_gogh_emb = embed_prompts(df["not_gogh"].tolist())

concept_vector = np.mean(van_gogh_emb - not_gogh_emb, axis=0)
np.save("checkpoints/vangogh_vector.npy", concept_vector)

print(f"Saved concept vector: shape {concept_vector.shape} (expect (77, 768))")
print(f"L2 norm: {np.linalg.norm(concept_vector):.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Saved concept vector: shape (77, 768) (expect (77, 768))
L2 norm: 128.9300


Inverse prompt for 10 target prompts we used for generation before

Importing libraries

In [5]:
import json
import random
import numpy as np
from pathlib import Path

parameters of gene-algo, unchanged

In [6]:
population_size = 200
generation = 3000
mutateRate = 0.25
crossoverRate = 0.5
length = 16
cof = 3

In [7]:
concept_vector = np.load("checkpoints/vangogh_vector.npy")
target_prompts = json.loads(Path("prompts/target_prompts.json").read_text())

In [10]:
def fitness(population, target_embed):
    tokens = torch.cat(population, 0).to("cuda")
    with torch.no_grad():
        embed = text_encoder(tokens)[0]
    losses = ((target_embed - embed) ** 2).sum(dim=(1, 2))
    return losses.cpu().numpy()

def crossover(parents, rate):
    new = []
    for i in range(len(parents)):
        new.append(parents[i])
        if random.random() < rate:
            j = np.random.randint(0, len(parents))
            point = np.random.randint(1, length + 1)
            new.append(torch.cat((parents[i][:, :point], parents[j][:, point:]), 1))
            new.append(torch.cat((parents[j][:, :point], parents[i][:, point:]), 1))
    return new
def mutate(population, rate):
    for i in range(len(population)):
        if random.random() < rate:
            idx = np.random.randint(1, length + 1)
            value = np.random.randint(1, 49406)
            population[i][:, idx] = value
    return population


Following block took over an hour for 10 target prompts, beware

In [12]:
recovered=[]
for prompt_idx, prompt in enumerate(target_prompts):
    print(f"\n=== Target {prompt_idx}: '{prompt[:60]}...' ===")

    text_input = tokenizer(prompt, padding="max_length", max_length=tokenizer.model_max_length,
                           truncation=True, return_tensors="pt")
    with torch.no_grad():
        target_embed = text_encoder(text_input.input_ids.to("cuda"))[0] + \
                       cof * torch.from_numpy(concept_vector).to("cuda")
    target_embed = target_embed.detach().clone()

    population = [
        torch.cat((
            torch.tensor([[49406]]),                                # BOS
            torch.randint(low=1, high=49406, size=(1, length)),     # variable tokens
            torch.full((1, 76 - length), 49407),                    # EOS padding
        ), 1)
        for _ in range(population_size)
    ]

    # GA loop
    for step in range(generation):
        scores = fitness(population, target_embed)
        order = np.argsort(scores)
        population = [population[i] for i in order][:population_size // 2]
        if step != generation - 1:
            population = mutate(crossover(population, crossoverRate), mutateRate)
        if step % 250 == 0:
            print(f"  Iter {step+1:>4}: min loss = {scores[order[0]]:.3f}")

    # Decode best candidate's 16 variable tokens back to text
    inv_prompt = tokenizer.decode(population[0][0][1:length + 1])
    print(f"  Recovered: {inv_prompt}")
    recovered.append(inv_prompt)

Path("prompts/recovered_prompts.json").write_text(json.dumps(recovered, indent=2))
print(f"\nSaved {len(recovered)} recovered prompts to prompts/recovered_prompts.json")


=== Target 0: 'Vincent van Gogh self-portrait, oil painting, post-impressio...' ===
  Iter    1: min loss = 304116.312
  Iter  251: min loss = 192367.750
  Iter  501: min loss = 176762.156
  Iter  751: min loss = 172852.719
  Iter 1001: min loss = 172688.984
  Iter 1251: min loss = 172506.781
  Iter 1501: min loss = 172351.219
  Iter 1751: min loss = 171447.812
  Iter 2001: min loss = 170498.469
  Iter 2251: min loss = 170215.312
  Iter 2501: min loss = 169242.125
  Iter 2751: min loss = 168591.094
  Recovered: roberchristopangeich gogh vu style paintingbbcrasurrealism following minor vina ssionist flaws frigh

=== Target 1: 'Vincent van Gogh self-portrait with a bandaged ear, oil pain...' ===
  Iter    1: min loss = 316810.688
  Iter  251: min loss = 225014.062
  Iter  501: min loss = 186888.188
  Iter  751: min loss = 178883.203
  Iter 1001: min loss = 176337.156
  Iter 1251: min loss = 175414.688
  Iter 1501: min loss = 174694.938
  Iter 1751: min loss = 173987.344
  Iter 2001: min

In [13]:
#display targets and their recovered counterparts
import json
from pathlib import Path

recovered = json.loads(Path("prompts/recovered_prompts.json").read_text())
target = json.loads(Path("prompts/target_prompts.json").read_text())
for i, (t, r) in enumerate(zip(target, recovered)):
    print(f"\n=== {i} ===")
    print(f"Target:    {t}")
    print(f"Recovered: {r}")


=== 0 ===
Target:    Vincent van Gogh self-portrait, oil painting, post-impressionist, thick swirling brushstrokes, masterpiece
Recovered: roberchristopangeich gogh vu style paintingbbcrasurrealism following minor vina ssionist flaws frigh

=== 1 ===
Target:    Vincent van Gogh self-portrait with a bandaged ear, oil painting, green coat, fur hat, post-impressionist
Recovered: physicians vignekettlice proves cawx nev🇮sketched hahah gogh inna ultimatluxembourportraitviola

=== 2 ===
Target:    a portrait of an old peasant man with a weathered face, Vincent van Gogh style, oil on canvas, warm earth tones, museum quality
Recovered: normally amusement transforms pressicialdepicting vincent esque �gallons gogh portrait dans vangogh �dged

=== 3 ===
Target:    a portrait of a postman with a thick dark beard in a blue uniform, painted by Vincent van Gogh, post-impressionist oil painting
Recovered: invitgave vangogh impressionist starring worker ;) aka vincenmaar josef gogh tedly apiseveedshee